In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import time

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
from keras.losses import CategoricalCrossentropy
import csv

from loss import *
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator as DAE_OptDG
from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
from plotting import *


2026-10-29 17:48:21.780647: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import models
import importlib
importlib.reload(models)
from models import *

thresholds = [1,240,800]

xformer = Create_2head_xformer_model(input_shape=(16,16,2),   
                             patch_size=(3,4),        
                             embed_dim=64,           
                             num_heads=4,            
                             ff_dim=128,              
                             num_layers=4,            
                             dropout=0.1,        
                             threshold_offset=0,
                             head_dims=(256,128,64),
                             n_bits=2,
                             initial_thresholds=thresholds,
                             final_outputs=14        
                            )
xformer.summary()

2026-10-29 17:48:24.940576: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-10-29 17:48:24.991054: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-10-29 17:48:24.991552: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-10-29 17:48:24.998611: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 raw_input (InputLayer)         [(None, 16, 16, 2)]  0           []                               
                                                                                                  
 soft_quantizer_output (SoftQua  (None, 16, 16, 2)   4           ['raw_input[0][0]']              
 ntizeLayer)                                                                                      
                                                                                                  
 patch_extractor (PatchExtracto  (None, None, 24)    0           ['soft_quantizer_output[0][0]']  
 r)                                                                                               
                                                                                              

In [3]:
import mdmm
importlib.reload(mdmm)
from mdmm import *

MDMM_SCALE = 1e0
MDMM_DAMPING = 10.0
MDMM_CONSTRAINT_SAMPLES = None  # full batch -- 40GB slice, no OOM risk (see campaign 4)
# MDMM_MIN_CORR = {"x": 0.5, "y": 0.5, "cotA": 0.5, "cotB": 0.5}
# MDMM_OUTPUT_COLUMNS = {"x": 0, "y": 2, "cotA": 4, "cotB": 6}
# MDMM_LABEL_COLUMNS = {"x": 0, "y": 1, "cotA": 2, "cotB": 3}

constraints = [
            CCE_EqConstraint(value = 9e4, scale=MDMM_SCALE, damping=MDMM_DAMPING)
        
            # MinCorrConstraint(column=MDMM_OUTPUT_COLUMNS[p], label_column=MDMM_LABEL_COLUMNS[p],
            #                   min_value=MDMM_MIN_CORR[p],
            #                   scale=MDMM_SCALE, damping=MDMM_DAMPING, name=f"corr_{p}")
            # for p in ("x", "y", "cotA", "cotB")
        ]

model = MDMM(xformer, constraints, constraint_samples=MDMM_CONSTRAINT_SAMPLES, name="mdmm_xformer")

model.compile(
    # optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    optimizer=tf.keras.optimizers.Nadam(learning_rate=2e-3, clipnorm=1.0),
    # loss = CategoricalCrossentropy(reduction = 'sum', from_logits=True)
    loss = custom_loss
    # loss={
    #     "reg_output": custom_loss,
    #     "SQAutoEncoder_output": None                          # Ignores this branch during loss calculation
    # }
    # run_eagerly=True
)

In [4]:
import importlib
import mdmm_DataGenerator
# import dataset_utils
# import SoftQuantizeLayer
# import AnnealingScheduler

importlib.reload(mdmm_DataGenerator)
# importlib.reload(dataset_utils)
# importlib.reload(SoftQuantizeLayer)
# importlib.reload(AnnealingScheduler)

from AnnealingScheduler import AnnealingScheduler
from mdmm_DataGenerator import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
#from dataprep import *
# from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *

In [5]:
dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "tfrecords")

# dataset_dir_train = os.path.join(dataset_base_dir, "/dataset_3sr_16x16_50x12P5_parquets", 'train/')
# dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets", 'test/')

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'train_contained/')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_centeredIncidence_parquets", 'test_contained/')

# tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'SQDAE_80eN_catgoricalOut_Target_0_7_23_134_404_867_1748')
# tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_80eN_catgoricalOut_Target_0_7_23_134_404_867_1748')

tfrecords_dir_train   = os.path.join(tfrecords_base_dir, "TFR_train",'mdmm_3srCI_80eN_2bitThresh0_30_845')
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'mdmm_3srCI_80eN_2bitThresh0_30_845')

# tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3sr_16x16_80eN_raw')
# tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val",'3sr_16x16_80eN_raw')


print(dataset_dir_val)
batch_size = 5000
val_batch_size = 5000
train_file_size = 80
val_file_size = 20

/uscms/home/bweiss/nobackup/smart-pixels/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/test_contained/


In [ ]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_dir_val,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= False,
#     log_scale = False,
#     select_contained = False,
#     noise = [0, 80], #-1, #[0,240],
#     charge_thresholds= [0, 30, 845],
#     bitwise_categorical = True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), #
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,
#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# for out in validation_generator[0]:
#     print(type(out))
#     if isinstance(out, dict):
#         print('containing: ', out.keys())


# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_dir_train,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    # optimize_batch_size = True,
    file_count = train_file_size,
    to_standardize= False,
    select_contained = True,
    log_scale = False,
    labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
    input_shape = (2,16,16), # (20,13,21),
    transpose = (0,2,3,1),
    shuffle = False, # True 
    noise = [0, 80],
    charge_thresholds= [0, 30, 845],
    bitwise_categorical = True,
    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 2
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))


In [6]:
#### Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=False,
    bitwise_categorical = True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=False,
    bitwise_categorical = True,
)

Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_val/mdmm_3srCI_80eN_2bitThresh0_30_845/metadata.json


Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_train/mdmm_3srCI_80eN_2bitThresh0_30_845/metadata.json


In [7]:
from datetime import datetime
tname = 'mdmmTransformer_2bit_80eN_2bitThresh0_30_845_TEST'#0_7_23_134_404_867_1748'
fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./mdmm_trainings/model-{fingerprint}-{tname}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.4f}-v{val_loss:.4f}.hdf5'
print("fingerprint: ", fingerprint)

fingerprint:  351e88f6


In [8]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

early_stopping_patience = 1000

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort(key=lambda x: os.path.getmtime(os.path.join(base_dir,x)))
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

class SoftQuantizeLoggerCallback(Callback):
    def __init__(self, log_filepath, layer_name="soft_quantizer_output"):
        super().__init__()
        self.log_filepath = log_filepath
        self.layer_name = layer_name
        self.header_written = False

    def on_train_begin(self, logs=None):
        os.makedirs(os.path.dirname(self.log_filepath), exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        try:
            layer = self.model.get_layer(self.layer_name)
            if not hasattr(layer, 'n_bits'):
                 print(f"\nWarning: Layer '{self.layer_name}' is not a SoftQuantizeLayer. Skipping logging.")
                 return
        except ValueError:
            print(f"\nWarning: Layer '{self.layer_name}' not found in the model. Skipping logging.")
            return

        if not self.header_written:
            num_levels = layer.num_levels
            num_thresholds = num_levels - 1
            
            header = ['epoch', 'k']
            # header.extend([f'level_{i}' for i in range(num_levels)])
            header.extend([f'threshold_{i}' for i in range(num_thresholds)])

            # header.append('raw_first_level')
            # header.extend([f'raw_log_level_delta_{i}' for i in range(num_levels - 1)])
            # header.append('raw_first_threshold')
            # if hasattr(layer, 'log_threshold_deltas'):
            #     header.extend([f'raw_log_threshold_delta_{i}' for i in range(num_thresholds - 1)])

            with open(self.log_filepath, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(header)
            self.header_written = True

        k_val = layer.k.numpy().item()
        # levels = layer.levels.numpy().tolist()
        thresholds = layer.thresholds.numpy().tolist()
        
        # first_level = layer.first_level.numpy().item()
        # log_level_deltas = layer.log_level_deltas.numpy().tolist()
        # first_threshold = layer.first_threshold.numpy().item()
        
        row_data = [epoch, k_val]
        # row_data.extend(levels)
        row_data.extend(thresholds)
        # row_data.append(first_level)
        # row_data.extend(log_level_deltas)
        # row_data.append(first_threshold)
        
        # if hasattr(layer, 'log_threshold_deltas'):
        #     log_threshold_deltas = layer.log_threshold_deltas.numpy().tolist()
        #     row_data.extend(log_threshold_deltas)
        
        with open(self.log_filepath, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(row_data)


csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=400.0, 
    verbose=1      
)

quantizer_logger = SoftQuantizeLoggerCallback(
    log_filepath=f"{base_dir}/soft_quantizer_state_log.csv", # New, more descriptive filename
    layer_name="soft_quantizer_output"
)

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)


In [ ]:
history = None
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger, scheduler_callback, quantizer_logger
                  ],
        epochs=5000,
        shuffle=False,
        verbose=1
    )

2026-10-29 17:48:49.809992: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-10-29 17:48:49.897369: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8800
2026-10-29 17:48:50.015193: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-10-29 17:48:50.016573: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-10-29 17:48:50.016612: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2026-10-29 17:48:50.018512: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-10-29 17:48:50.018619: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to


Epoch 1: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3133, 240.3133, 800.3134
	Tau: [120.15664 399.50003 560.00006]
Epoch 1/5000


2026-10-29 17:49:03.225698: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x555897850b00


84/84 [==============================] - ETA: 0s - loss: 299690112.0000 - loss_obj: 99228.3671 - pen_cce__eq_constraint: 17169245162510.4766    
Epoch 1: val_loss improved from inf to 102134.61719, saving model to ./mdmm_trainings/model-351e88f6-mdmmTransformer_2bit_80eN_2bitThresh0_30_845_TEST-checkpoints/weights.01-t299690112.0000-v102134.6172.hdf5
84/84 [==============================] - 88s 824ms/step - loss: 299690112.0000 - loss_obj: 99279.9902 - pen_cce__eq_constraint: 16967257567498.5410 - val_loss: 102134.6172

Epoch 2: Annealing 'k' set to 1.0000
	Levels: 0.0000, 1.0000, 2.0000, 3.0000
	Thresholds: 1.3694, 249.2576, 824.9570
	Tau: [124.62879 411.7938  575.69946]
Epoch 2/5000
84/84 [==============================] - ETA: 0s - loss: 78688501760.0000 - loss_obj: 102582.1387 - pen_cce__eq_constraint: 3285205378635.6191 
Epoch 2: val_loss did not improve from 102134.61719
84/84 [==============================] - 38s 454ms/step - loss: 78688501760.0000 - loss_obj: 102594.3056 - pen